# Week 15, Day 2: Expectation, variance and the binomial: quantal synaptic release

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w015/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Moments from a PMF, and the covariance term

Write `mean_var(pmf)` returning the mean and variance of a PMF given as an array indexed by value. Then, for two neurons that share a gain, check linearity of expectation and compute `cov_ab` and `var_sum` = Var(a + b).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1954)

def mean_var(pmf):
    k = np.arange(len(pmf))
    mean = ...
    var = ...
    return mean, var

g = rng.gamma(5.0, 1 / 5.0, 20000)
a, b = rng.poisson(3 * g), rng.poisson(5 * g)
cov_ab = ...
var_sum = np.var(a + b)
print("mean_var([0.2, 0.5, 0.3]) =", mean_var(np.array([0.2, 0.5, 0.3])))
print(f"Var(a) + Var(b) = {np.var(a) + np.var(b):.3f},  2 Cov = {2 * cov_ab:.3f},  Var(a + b) = {var_sum:.3f}")

In [ ]:
import numpy as np
def test_moments():
    "mean_var returns the mean and variance of a PMF"
    m, v = mean_var(np.array([0.2, 0.5, 0.3]))
    assert abs(m - 1.1) < 1e-12 and abs(v - 0.49) < 1e-12

def test_linear():
    "the mean of the summed counts equals the sum of the means, even though the neurons are dependent"
    assert abs(np.mean(a + b) - (np.mean(a) + np.mean(b))) < 1e-9

def test_cov():
    "Var(a + b) equals Var(a) + Var(b) + 2 Cov(a, b), and the covariance is clearly positive"
    assert abs(var_sum - (np.var(a) + np.var(b) + 2 * cov_ab)) < 1e-6 and cov_ab > 0.5

test_moments()
test_linear()
test_cov()
print("All checks passed")

### Exercise: Find n and q from the variance-mean parabola

Simulate 400 responses at each of six release probabilities from a synapse with n = 8 sites and quantal size q = 0.4 mV (each quantum varies by 20%). Measure the mean and variance at each p, then fit Var = a·mean + b·mean² by least squares and recover `q_hat` = a and `n_hat` = −1/b.

In [ ]:
n_sites, q, cv = 8, 0.4, 0.2
ps = np.array([0.05, 0.15, 0.3, 0.5, 0.7, 0.85])

def responses(p, trials=400):
    k = rng.binomial(n_sites, p, trials)                              # quanta released on each trial
    quanta = [q * np.sum(1 + cv * rng.normal(size=kk)) for kk in k]  # each quantum varies a little
    return np.array(quanta) + 0.02 * rng.normal(size=trials)          # recording noise

means, variances = [], []
for p in ps:
    v = responses(p)
    ...
means, variances = np.array(means), np.array(variances)
a_coef, b_coef = ...
q_hat, n_hat = a_coef, -1 / b_coef
print("means:", means.round(3), " variances:", variances.round(4))
print(f"estimated q = {q_hat:.3f} mV (true 0.4), n = {n_hat:.2f} (true 8)")
m = np.linspace(0, means.max() * 1.1, 100)
plt.plot(means, variances, "o"); plt.plot(m, q_hat * m - m ** 2 / n_hat); plt.xlabel("mean (mV)"); plt.ylabel("variance (mV²)"); plt.show()

In [ ]:
import numpy as np
def test_points():
    "means and variances are measured at all six release probabilities"
    assert len(means) == 6 and len(variances) == 6 and np.all(np.diff(means) > 0)

def test_fit():
    "the fitted n is within 1.5 of 8 and the fitted q within 15% of 0.4"
    assert abs(n_hat - 8) < 1.5 and abs(q_hat - 0.4) / 0.4 < 0.15

test_points()
test_fit()
print("All checks passed")

### Exercise: The method of failures

At p = 0.05, over 2,000 trials, count failures (responses below q/2) to get `m_failures` = −ln(fraction of failures), and compare with `m_direct` = mean response / q.

In [ ]:
v_low = responses(0.05, 2000)
fail_frac = ...
m_failures = ...
m_direct = ...
print(f"failures: {fail_frac:.3f};  m from failures {m_failures:.3f},  m from the mean {m_direct:.3f},  true n p = {n_sites * 0.05}")

In [ ]:
import numpy as np
def test_two_estimates():
    "the failure estimate and the direct estimate of m agree with each other and with np to within 0.1"
    assert abs(m_failures - m_direct) < 0.1 and abs(m_failures - n_sites * 0.05) < 0.1

test_two_estimates()
print("All checks passed")